# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Per the framing in Week 2 (`is_declining_label` is a yes/no label derived from an observed rule),
and the task-type table in the `training-honest-models` skill, I'm training three methods in
increasing complexity and comparing all three against the frozen Week-4 baseline on the same
split:

- **Logistic Regression** -- the readable linear floor; if it wins, complexity wasn't needed.
- **Decision Tree** (`max_depth=5, min_samples_leaf=50`) -- still printable and explainable, one
  step up from the baseline's fixed weights.
- **Random Forest** (`max_depth=10, min_samples_leaf=25, 200 trees`) -- the complexity is only
  justified if it earns its keep over the tree, checked in section 3, not assumed.

Depth/leaf limits on the tree and forest match the reference pipeline's regularization
(`scripts/03_train_model.py`) rather than unlimited depth -- unregularized trees memorize
individual clients and look artificially strong in-sample, then generalize worse to a client
they've never seen.


In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd, numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, percentile_rank

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
fv = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").copy()

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    fv[f"log_{col}"] = np.log1p(fv[col].fillna(0))
fv["is_declining_label"] = fv["trend_direction"].str.lower().eq("down").astype(int)
for col in MODEL_NUMERIC_FEATURES:
    fv[col] = pd.to_numeric(fv[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for col in MODEL_CATEGORICAL_FEATURES:
    fv[col] = fv[col].fillna("unknown").astype(str)

X = pd.get_dummies(fv[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES], columns=MODEL_CATEGORICAL_FEATURES)
y = fv["is_declining_label"]
print("Feature matrix:", X.shape, "| declining rate:", round(y.mean(), 3))


Feature matrix: (30000, 52) | declining rate: 0.542


## 2. Split design

**Client-grouped holdout, not random-row holdout.** Pages from the same client share hidden
character -- templates, content strategy, internal linking -- so a random row split would let the
model partially memorize a client it has already seen elsewhere in training, and the score would
be optimistic. Holding out ~20% of the 32 *clients* (not rows) and testing only on clients the
model never trained on is the honest version of "will this work on a new client?"

**The catch, and why it matters here:** 32 clients is a small population to draw 20% from -- that's
only 6 clients in the holdout. I checked whether that makes the result sensitive to *which* 6
clients land in the holdout by rerunning the same Random Forest across 3 different holdout seeds:
Precision@50 ranged **0.68 -> 0.84 -> 0.98** depending on the draw. All three seeds beat the
baseline comfortably, so the *direction* of the finding is solid -- but the exact number is a
property of the split, not just the model, and reporting that honestly matters more than picking
the most flattering seed. Section 3 uses `seed=42` (matching this repo's convention throughout)
and states that choice up front rather than silently.


In [2]:
def make_client_split(frame, seed):
    rng = np.random.default_rng(seed)
    unique_clients = frame["client_id"].drop_duplicates().to_numpy()
    shuffled = rng.permutation(unique_clients)
    n_holdout = max(1, int(round(len(shuffled) * 0.2)))
    holdout = set(shuffled[:n_holdout])
    return frame["client_id"].isin(holdout).to_numpy(), n_holdout

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:min(k, len(labels))]].mean()

print("Seed sensitivity check (Random Forest, same hyperparameters each time):")
for seed in (42, 7, 123):
    is_test_probe, n_ho = make_client_split(fv, seed)
    X_tr, X_te = X[~is_test_probe], X[is_test_probe]
    y_tr, y_te = y[~is_test_probe], y[is_test_probe]
    rf_probe = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                       min_samples_leaf=25, n_estimators=200,
                                       n_jobs=-1, random_state=42).fit(X_tr, y_tr)
    p = rf_probe.predict_proba(X_te)[:, 1]
    print(f"  seed={seed:<4} holdout_clients={n_ho}  test_rows={len(X_te):<5}  "
          f"test_base_rate={y_te.mean():.3f}  P@50={precision_at_k(p, y_te.values, 50):.3f}")

# the split used for every result from here on
is_test, n_holdout = make_client_split(fv, 42)
X_train, X_test = X[~is_test], X[is_test]
y_train, y_test = y[~is_test], y[is_test]
print(f"\nUsing seed=42: {n_holdout} holdout clients, {len(X_test)} test rows, "
      f"test base rate {y_test.mean():.3f}")


Seed sensitivity check (Random Forest, same hyperparameters each time):


  seed=42   holdout_clients=6  test_rows=2325   test_base_rate=0.391  P@50=0.680


  seed=7    holdout_clients=6  test_rows=4280   test_base_rate=0.611  P@50=0.840


  seed=123  holdout_clients=6  test_rows=3090   test_base_rate=0.549  P@50=0.980

Using seed=42: 6 holdout clients, 2325 test rows, test base rate 0.391


## 3. Train + compare vs my baseline

Same held-out clients, same precision@K / ROC-AUC metrics as everywhere else in this project. The
**Week-4 baseline is recomputed on this exact test slice** (not the full-population numbers from
Week 4) so the comparison is apples-to-apples -- a rule scored against rows it was implicitly
"tuned" against (all 30,000) isn't a fair opponent for a model scored strictly on unseen clients.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr = Pipeline([("scaler", StandardScaler()),
               ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
lr.fit(X_train, y_train)
p_lr = lr.predict_proba(X_test)[:, 1]

dt = DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=42)
dt.fit(X_train, y_train)
p_dt = dt.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                             n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]

# Week-4 baseline, recomputed on the SAME test rows
fv["visibility_score"] = percentile_rank(np.log1p(fv["impressions_90d"]))
fv["freshness_risk_score"] = percentile_rank(fv["days_since_last_update"])
positioned = fv[(fv["avg_position"] > 0) & (fv["avg_position"] <= 50)]
tier_median_ctr = positioned.groupby("position_tier")["ctr"].median()
fv["expected_ctr"] = fv["position_tier"].map(tier_median_ctr).fillna(fv["ctr"])
fv["ctr_gap"] = (fv["expected_ctr"] - fv["ctr"]).clip(lower=0)
has_pos = (fv["avg_position"] > 0).astype(int)
fv["ctr_gap_score"] = percentile_rank(fv["ctr_gap"]) * has_pos
fv["baseline_action_score"] = (0.45 * fv["visibility_score"] + 0.35 * fv["freshness_risk_score"]
                                + 0.20 * fv["ctr_gap_score"]).clip(0, 1)
p_base = fv.loc[is_test, "baseline_action_score"].to_numpy()

print(f"{'method':<24}{'ROC-AUC':>10}{'P@20':>8}{'P@50':>8}")
print("-" * 50)
for name, p in [("baseline_action_score", p_base), ("logistic_regression", p_lr),
                ("decision_tree (d=5)", p_dt), ("random_forest", p_rf)]:
    auc = roc_auc_score(y_test, p)
    p20 = precision_at_k(p, y_test.values, 20)
    p50 = precision_at_k(p, y_test.values, 50)
    print(f"{name:<24}{auc:>10.3f}{p20:>8.3f}{p50:>8.3f}")
print(f"\ntest base rate (the floor every number above should beat): {y_test.mean():.3f}")


method                     ROC-AUC    P@20    P@50
--------------------------------------------------
baseline_action_score        0.646   0.400   0.420
logistic_regression          0.700   0.350   0.400
decision_tree (d=5)          0.742   0.650   0.660
random_forest                0.747   0.700   0.680

test base rate (the floor every number above should beat): 0.391


## 4. Errors and interpretation

**What the forest leans on:** `log_impressions_90d`, `days_with_impressions`, `avg_position`, and
`content_age_days` dominate the top of the importance table below -- all pre-decision, observable
signals, none label-adjacent. That's consistent with the Week-3 leakage check (the same signals
showed up there) and reassuring: the model isn't leaning on anything suspicious.

**Where it's wrong:** of the top-50 ranked test pages, 16 are false positives -- pages that look
exactly like decliners (visible, older, mid-pack position, thin CTR) but weren't in a >20% drop
this window. That's the same pattern the Week-4 baseline's weak picks showed: believable "not yet,
but watch it" pages, not nonsense picks. On the other side, the worst-scored true decliners are
pages with almost no signal at all -- 1-3 impressions in 90 days, `avg_position = 0` (no data). The
model correctly has nothing to go on for these; a page with almost no measurable activity isn't
something any method here can call "declining" versus "always was invisible" -- a real limit of
this label, not a model bug (see Week-3's data-limits note on `avg_position == 0`).


In [4]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 8 random forest feature importances:")
print(imp.head(8).round(3).to_string())

test_frame = fv.loc[is_test].copy()
test_frame["rf_score"] = p_rf
top50 = test_frame.sort_values("rf_score", ascending=False).head(50)
fp = top50[top50["is_declining_label"] == 0]
print(f"\nFalse positives in the RF top 50: {len(fp)} of 50")
print(fp[["content_id", "rf_score", "impressions_90d", "avg_position", "ctr", "days_since_last_update"]]
      .head(4).to_string(index=False))

fn = test_frame[test_frame["is_declining_label"] == 1].sort_values("rf_score").head(4)
print("\nWorst-scored true decliners (false negatives):")
print(fn[["content_id", "rf_score", "impressions_90d", "avg_position", "ctr", "days_since_last_update"]]
      .to_string(index=False))


Top 8 random forest feature importances:
log_impressions_90d      0.132
days_with_impressions    0.131
avg_position             0.111
content_age_days         0.091
age_tier_365+            0.038
word_count               0.037
log_clicks_90d           0.037
char_count               0.036

False positives in the RF top 50: 16 of 50
          content_id  rf_score  impressions_90d  avg_position  ctr  days_since_last_update
content_d2dffcc697a4  0.737431             5091          14.1 0.20                      20
content_00603b0349b4  0.735212             1076          25.6 0.09                      20
content_e55b8ab078b0  0.733797              369          21.8 0.00                      20
content_f5013794ba57  0.732604              881          15.7 0.00                      20

Worst-scored true decliners (false negatives):
          content_id  rf_score  impressions_90d  avg_position   ctr  days_since_last_update
content_28b4223f4e5f  0.081668                1           0.0  0.00     

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.